## Gold — Product Performance
Top sellers, slow movers, and revenue by category.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

### Read Daily Demand Gold Table

In [0]:
df_daily = spark.read.table("databricks_cat.gold.DailyDemand")

### Aggregate Product-Level KPIs

In [0]:
df_product_kpi = (
    df_daily
    .groupBy("product_id", "product_name", "category", "brand")
    .agg(
        sum("units_sold").alias("total_units_sold"),
        sum("daily_revenue").alias("total_revenue"),
        avg("return_rate_pct").alias("avg_return_rate_pct"),
        countDistinct("order_date").alias("active_selling_days")
    )
    .withColumn("revenue_per_day",
        when(col("active_selling_days") > 0,
             col("total_revenue") / col("active_selling_days")).otherwise(lit(0.0))
    )
)
display(df_product_kpi)

### Rank Products Within Category — Label Top Sellers & Slow Movers

In [0]:
window_units = Window.partitionBy("category").orderBy(desc("total_units_sold"))
window_rev   = Window.partitionBy("category").orderBy(desc("total_revenue"))
window_count = Window.partitionBy("category")

df_product_kpi = (
    df_product_kpi
    .withColumn("rank_by_units",     rank().over(window_units))
    .withColumn("rank_by_revenue",   rank().over(window_rev))
    .withColumn("total_in_category", count("product_id").over(window_count))
    .withColumn("product_label",
        when(col("rank_by_units") <= 3, lit("TOP_SELLER"))
        .when(col("rank_by_units") >= col("total_in_category") - 2, lit("SLOW_MOVER"))
        .otherwise(lit("NORMAL"))
    )
    .drop("total_in_category")
)
df_product_kpi.display()

### Revenue by Category Summary

In [0]:
df_category_revenue = (
    df_daily
    .groupBy("category")
    .agg(
        sum("daily_revenue").alias("total_category_revenue"),
        sum("units_sold").alias("total_category_units"),
        avg("return_rate_pct").alias("avg_category_return_rate"),
        countDistinct("product_id").alias("distinct_products")
    )
    .orderBy(desc("total_category_revenue"))
)
df_category_revenue.display()

### Save ProductPerformance → Gold

In [0]:
gold_perf_path = "abfss://gold@databricksetestorage.dfs.core.windows.net/ProductPerformance"

if spark.catalog.tableExists("databricks_cat.gold.ProductPerformance"):
    dt = DeltaTable.forName(spark, "databricks_cat.gold.ProductPerformance")
    (
        dt.alias("trg")
        .merge(df_product_kpi.alias("src"), "trg.product_id = src.product_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("✅ ProductPerformance upserted.")
else:
    df_product_kpi.write.format("delta").option("path", gold_perf_path).saveAsTable("databricks_cat.gold.ProductPerformance")
    print("✅ ProductPerformance created.")

### Save CategoryRevenue → Gold

In [0]:
(
    df_category_revenue
    .write.format("delta")
    .mode("overwrite")
    .option("path", "abfss://gold@databricksetestorage.dfs.core.windows.net/CategoryRevenue")
    .saveAsTable("databricks_cat.gold.CategoryRevenue")
)
print("✅ CategoryRevenue written successfully.")